# Lab 5 — sklearn Pipeline

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Goals

1. Combine **preprocessing** and **classification** in one `Pipeline`.
2. Use `ColumnTransformer` to apply different transforms to numeric vs categorical columns.
3. Fit and evaluate on the same train/test split as earlier labs (800 / 200).
4. Explain why preprocessing must live **inside** the pipeline (no data leakage).

> **Quick check:** steps `['preprocess', 'clf']` · test accuracy ≈ **0.64** · train 800 / test 200




## Why this matters <!-- cisco-doc-enrich-2026 -->

**Pipelines** bundle preprocessing + model so test data never leaks statistics from train.
One `fit` / `predict` call is what you deploy to production.

```
Raw row  -->  [imputer | scaler | encoder]  -->  classifier  -->  prediction
              \___________ Pipeline __________/
```


## Why a Pipeline?

| Problem | Without Pipeline | With Pipeline |
|---------|------------------|---------------|
| **Data leakage** | Fit scaler on full data, then split | Fit scaler on **train only** inside `pipe.fit()` |
| **Deployment** | Remember transform order by hand | `pipe.predict(new_row)` applies everything |
| **Cross-validation** | Easy to leak labels across folds | `cross_val_score(pipe, X, y)` is safe |

---

## 1. Load data and define feature groups

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

# cisco-output-ready
print(f"Setup OK — repo root: {GH_ROOT}")


Setup OK — repo root: C:\25-Trainings\2-Confirmed\10-June-26-AI-ML-Cisco\GH


**Step 2** — run the cell below and read the printed summary. <!-- cisco-split-debug-2026 -->

In [2]:
LENDING_CLUB_CSV = GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
CATEGORICAL_FEATURES = ["grade", "term"]

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv(LENDING_CLUB_CSV)
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X = df[feature_cols]
y = df["default"]

print(f"shape: {X.shape}")
print(f"default rate: {y.mean():.3f}")
display(X.head(3))

shape: (1000, 7)
default rate: 0.485


,loan_amnt,int_rate,annual_inc,dti,installment,grade,term
0,34498,14.50,35829,15.66,98.07,E,36 months
1,33265,6.75,105184,5.91,859.60,A,36 months
2,4012,6.48,139128,13.90,320.56,C,36 months


| Type | Columns |
|------|--------|
| Numeric | `loan_amnt`, `int_rate`, `annual_inc`, `dti`, `installment` |
| Categorical | `grade`, `term` |

### 1b. Cardinality of categoricals

In [3]:
for col in CATEGORICAL_FEATURES:
    print(f"{col}: {X[col].nunique()} unique →", X[col].value_counts().to_dict())


# cisco-debug-summary
print("Value counts — long tail categories may be omitted.")

grade: 7 unique → {'G': 161, 'C': 150, 'A': 149, 'E': 137, 'B': 137, 'F': 136, 'D': 130}
term: 2 unique → {'36 months': 509, '60 months': 491}
Value counts — long tail categories may be omitted.


---

## 2. Train/test split (same as Labs 2–4)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train size: {len(X_train)}, test size: {len(X_test)}")
print(f"train default rate: {y_train.mean():.3f}")
print(f"test default rate: {y_test.mean():.3f}")


train size: 800, test size: 200
train default rate: 0.485
test default rate: 0.485


---

## 3. Build `ColumnTransformer`

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

`handle_unknown='ignore'` prevents errors when a new category appears at scoring time.

---

## 4. Wrap in `Pipeline` and fit

In [6]:
pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

pipe.fit(X_train, y_train)
print(f"pipeline steps: {[name for name, _ in pipe.steps]}")


pipeline steps: ['preprocess', 'clf']

---

## 5. Predict and measure accuracy

In [7]:
y_pred = pipe.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"test accuracy: {accuracy:.4f}")
print(f"sample predictions (first 5): {y_pred[:5].tolist()}")


test accuracy: 0.6350
sample predictions (first 5): [0, 0, 1, 0, 0]


Adding categoricals (`grade`, `term`) typically lifts accuracy over the numeric-only model (~0.59).

---

## 6. Peek inside transformed features

In [8]:
X_train_transformed = pipe.named_steps["preprocess"].transform(X_train)
print(f"transformed train shape: {X_train_transformed.shape}")

ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
all_names = list(NUMERIC_FEATURES) + list(cat_names)
print(f"feature count after encoding: {len(all_names)}")
print("sample encoded names:", all_names[:8], "...")


transformed train shape: (800, 14)
feature count after encoding: 14
sample encoded names: ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'installment', 'grade_A', 'grade_B', 'grade_C'] ...


### 6b. One row through the pipeline

In [9]:
sample_row = X_test.iloc[[0]]
raw_pred = pipe.predict_proba(sample_row)[0, 1]
print(f"first test row P(default): {raw_pred:.4f}")


first test row P(default): 0.4629


---

## 7. Numeric-only baseline

In [10]:
pipe_numeric = Pipeline(
    steps=[
        ("preprocess", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
pipe_numeric.fit(X_train[NUMERIC_FEATURES], y_train)
acc_numeric = accuracy_score(y_test, pipe_numeric.predict(X_test[NUMERIC_FEATURES]))

print(f"numeric-only accuracy: {acc_numeric:.4f}")
print(f"full pipeline accuracy: {accuracy:.4f}")
print(f"lift from categoricals: {accuracy - acc_numeric:+.4f}")


numeric-only accuracy: 0.5900
full pipeline accuracy: 0.6350
lift from categoricals: +0.0450


---

## 8. Why leakage matters

In [11]:
# WRONG pattern (do not use in production):
# scaler_bad = StandardScaler().fit(X[NUMERIC_FEATURES])  # fit on ALL rows including test
# Then split — test statistics leaked into scaler
print("Pipeline fits preprocessors only on X_train inside pipe.fit() — safe.")


# cisco-debug-summary
print("Model fit complete.")

Pipeline fits preprocessors only on X_train inside pipe.fit() — safe.
Model fit complete.


### 8b. `cross_val_score` with pipeline

In [12]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
print(f"5-fold CV accuracy: {cv_scores.round(3)}")
print(f"mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


5-fold CV accuracy: [0.66  0.68  0.625 0.665 0.615]
mean: 0.6490 ± 0.0248


### 8c. Pipeline `predict_proba` on test

In [13]:
pipe_proba = pipe.predict_proba(X_test)[:, 1]
print(f"score range: {pipe_proba.min():.3f} – {pipe_proba.max():.3f}")


score range: 0.100 – 0.888


### 8d. Confusion matrix from pipeline

In [14]:
from sklearn.metrics import confusion_matrix

cm_pipe = confusion_matrix(y_test, y_pred)
print(cm_pipe)


[[69 34]
 [39 58]]


### 8e. Grade coefficient direction (via one-hot)

In [15]:
# After fit, inspect logistic coefficients on encoded matrix
n_features = pipe.named_steps["clf"].coef_.shape[1]
print(f"classifier sees {n_features} encoded features")


classifier sees 14 encoded features


### 8f. Serialize pipeline with joblib

In [16]:
import joblib

out_dir = GH_ROOT / "hands-on" / "03-classification-interpretation" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
pkl = out_dir / "loan_default_pipe.joblib"
joblib.dump(pipe, pkl)
loaded = joblib.load(pkl)
print("reload predict match:", (loaded.predict(X_test) == y_pred).all())


reload predict match: True


### 8g. Single-row scoring API shape

In [17]:
new_app = X_test.iloc[[10]]
print("input columns:", list(new_app.columns))
print("prediction:", pipe.predict(new_app)[0])
print("P(default):", round(pipe.predict_proba(new_app)[0, 1], 4))


input columns: ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'installment', 'grade', 'term']
prediction: 0
P(default): 0.4758


### 8h. Term 60 months vs 36 months default rate

In [18]:
print(df.groupby("term")["default"].mean().round(3))


# cisco-debug-summary
print("Groupby complete — compare categories in the table above.")

term
36 months    0.497
60 months    0.473
Name: default, dtype: float64
Groupby complete — compare categories in the table above.


### 8i. `get_feature_names_out` full list

In [19]:
feat_names = pipe.named_steps["preprocess"].get_feature_names_out()
print(f"total features: {len(feat_names)}")
print(list(feat_names))


total features: 14
['num__loan_amnt', 'num__int_rate', 'num__annual_inc', 'num__dti', 'num__installment', 'cat__grade_A', 'cat__grade_B', 'cat__grade_C', 'cat__grade_D', 'cat__grade_E', 'cat__grade_F', 'cat__grade_G', 'cat__term_36 months', 'cat__term_60 months']


### 8j. Training accuracy from pipeline

In [20]:
train_acc = accuracy_score(y_train, pipe.predict(X_train))
print(f"train accuracy: {train_acc:.4f}, test: {accuracy:.4f}")


train accuracy: 0.6663, test: 0.6350


### 8k. `set_params` — change regularization C

In [21]:
pipe.set_params(clf__C=0.1)
pipe.fit(X_train, y_train)
acc_c01 = accuracy_score(y_test, pipe.predict(X_test))
print(f"accuracy with C=0.1: {acc_c01:.4f}")
pipe.set_params(clf__C=1.0)
pipe.fit(X_train, y_train)


# cisco-debug-summary
print("Model fit complete.")

accuracy with C=0.1: 0.6300


Model fit complete.


### 8l. Missing value check

In [22]:
print("missing in X:", X.isna().sum().sum())


missing in X: 0


### 8m. Grade default rates in raw data

In [23]:
print(df.groupby("grade")["default"].mean().round(3))


# cisco-debug-summary
print("Groupby complete — compare categories in the table above.")

grade
A    0.289
B    0.365
C    0.440
D    0.485
E    0.555
F    0.566
G    0.683
Name: default, dtype: float64
Groupby complete — compare categories in the table above.


### 8n. Pipeline steps introspection

In [24]:
for name, step in pipe.steps:
    print(name, "→", type(step).__name__)


preprocess → ColumnTransformer
clf → LogisticRegression


### 8o. Sample transformed row

In [25]:
row_t = pipe.named_steps["preprocess"].transform(X_test.iloc[[0]])
dense = row_t.toarray() if hasattr(row_t, "toarray") else row_t
print("transformed shape:", dense.shape)
print("first 8 values:", np.round(dense[0, :8], 3))


transformed shape: (1, 14)
first 8 values: [-0.879  0.795  0.801  0.757 -1.346  1.     0.     0.   ]


---

## 9. Try it yourself

List the output feature names after one-hot encoding `grade` and `term`.

In [26]:
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
names = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
print(f"total encoded categoricals: {len(names)}")
print(list(names))


total encoded categoricals: 9
['grade_A', 'grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F', 'grade_G', 'term_36 months', 'term_60 months']


---

## 10. Checkpoint summary

In [27]:
step_names = [name for name, _ in pipe.steps]
assert step_names == ["preprocess", "clf"]
assert len(X_train) == 800 and len(X_test) == 200
assert abs(accuracy - 0.6350) < 0.02
print("Numbers match — you're good.")



Numbers match — you're good.


## Feature selection (course topic)

Feature **engineering** creates columns; feature **selection** keeps the most predictive subset.

In [28]:
from sklearn.feature_selection import SelectKBest, f_classif

num_cols = NUMERIC_FEATURES
cat_cols = CATEGORICAL_FEATURES
X_all = df[num_cols + cat_cols]
y_all = df["default"]
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

selector = SelectKBest(score_func=f_classif, k=3)
selector.fit(X_tr[num_cols], y_tr)
scores = pd.DataFrame({"feature": num_cols, "score": selector.scores_}).sort_values("score", ascending=False)
print("top numeric features by ANOVA F-score:")
print(scores.round(2).to_string(index=False))


top numeric features by ANOVA F-score:
    feature  score
   int_rate  34.47
        dti  19.33
installment   4.85
  loan_amnt   1.26
 annual_inc   0.23


---

## Reflection questions

1. What would go wrong if you called `StandardScaler().fit(X)` on the full dataset before `train_test_split`?
2. Why is `handle_unknown='ignore'` useful when deploying to production?
3. How would you add `GridSearchCV` to tune `C` without leaking test data?
4. How much lift did `grade` and `term` add over numeric-only?
